# Infinitive Prediction with Masked Language Modeling

This notebook evaluates whether a masked language model predicts infinitives that correspond to the collexemes most strongly associated with a modal construction.

The task masks the infinitive in modal + infinitive constructions and compares the model's predicted ranking with the ranking obtained from Simple Collexeme Analysis.

The notebook expects as input:
- Sketch Engine KWIC exports for the target modal construction;
- simple collexeme analysis output from the collostructional analysis of each modal construction.

## Setup

In [ ]:
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install transformers accelerate pandas rbo unidecode tqdm scipy

In [ ]:
import os
import re
import csv
import json
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModel
from unidecode import unidecode
from rbo import RankingSimilarity

import matplotlib.pyplot as plt

In [ ]:
def rbo_ext_score(list1, list2, p=0.95):
    """Return extrapolated RBO score."""
    res = RankingSimilarity(list1, list2).rbo(p=p)
    return float(res["ext"] if isinstance(res, dict) else res)

## Configuration

In [ ]:
KWIC_CSV = "data/volere_inf_raw.csv"
GOLD_CSV = "data/collvol_out.csv"

OUT_RANK = "output/forward_volere_ranking.csv"
OUT_SCORES = "output/forward_volere_scores.json"
OUT_FILLED = "output/forward_volere_filled.csv"
OUT_RANK_FULL = "output/rank_full_vocab_vol.csv"

MLM_NAME = "indigo-ai/BERTino"
RANDOM_SEED = 13
TARGET_N_SENT = 10000
MAX_PER_VERB_CAP = 0
RBO_P = 0.95
BATCH_SIZE = 32

device = "cuda" if torch.cuda.is_available() else "cpu"

tok = AutoTokenizer.from_pretrained(MLM_NAME)
mlm = AutoModelForMaskedLM.from_pretrained(MLM_NAME).to(device).eval()

MASK_TOKEN = tok.mask_token
MASK_ID = tok.mask_token_id

## Reading and reconstructing Sketch Engine concordance data

In [ ]:
# Read semicolon-separated outer CSV and keep the field containing the inner CSV row.
outer_rows = []

with open(KWIC_CSV, "r", encoding="utf-8-sig", newline="") as f:
    outer_reader = csv.reader(f, delimiter=";", quotechar='"', doublequote=True)

    for row in outer_reader:
        if not row:
            continue

        outer_rows.append(row[0])

# Parse the inner comma-separated CSV records.
inner_rows = []

for s in outer_rows:
    reader = csv.reader([s], delimiter=",", quotechar='"', doublequote=True)

    for fields in reader:
        inner_rows.append(fields)

if not inner_rows:
    raise ValueError("No data parsed from the nested CSV file.")

header = [h.strip() for h in inner_rows[0]]
data = inner_rows[1:]

df = pd.DataFrame(data, columns=header)

print("Loaded nested CSV:", df.shape)
print("Columns:", df.columns.tolist())

In [ ]:
def pick_col(df, target):
    """Find a column by exact or partial case-insensitive match."""
    target = target.lower()

    for c in df.columns:
        if c.strip().lower() == target:
            return c

    for c in df.columns:
        if target in c.strip().lower():
            return c

    raise ValueError(f"Column '{target}' not found. Got: {list(df.columns)}")


left_col = pick_col(df, "left")
kwic_col = pick_col(df, "kwic")
right_col = pick_col(df, "right")

In [ ]:
START_TAG = re.compile(r"<s[^>]*>", re.I)
END_TAG = re.compile(r"</s>", re.I)
GENERIC_TAG = re.compile(r"<[^>]+>")

def after_last_start_tag(s: str) -> str:
    s = str(s)
    last = None

    for m in START_TAG.finditer(s):
        last = m

    return s[last.end():] if last else s


def before_first_end_tag(s: str) -> str:
    s = str(s)
    m = END_TAG.search(s)

    return s[:m.start()] if m else s


def strip_all_tags(s: str) -> str:
    return GENERIC_TAG.sub("", str(s))


def split_tokens_with_spans(s: str):
    """Return whitespace tokens and their character spans."""
    toks = re.findall(r"\S+", s)
    spans = []
    i = 0

    for t in toks:
        j = s.find(t, i)

        if j < 0:
            return [], [], s

        spans.append((j, j + len(t)))
        i = j + len(t)

    return toks, spans, s


def smart_concat(a: str, b: str) -> str:
    """Concatenate text segments while preserving punctuation spacing."""
    a = "" if a is None else str(a)
    b = "" if b is None else str(b)

    if not a:
        return b
    if not b:
        return a

    last = a[-1]
    first = b[0]

    if last.isspace() or first.isspace():
        return a + b
    if last in "([«„“" or first in ",.;:!?)]»”":
        return a + b
    if last == "'":
        return a + b

    return a + " " + b

## Masking the infinitive

In [ ]:
rows = []

for ridx, r in df.iterrows():
    left_raw = str(r[left_col])
    kwic_raw = str(r[kwic_col])
    right_raw = str(r[right_col])

    left_part = after_last_start_tag(left_raw)
    right_part = before_first_end_tag(right_raw)

    left_part = strip_all_tags(left_part)
    kwic_str = strip_all_tags(kwic_raw)
    right_part = strip_all_tags(right_part)

    kw_toks, kw_spans, _ = split_tokens_with_spans(kwic_str)

    if len(kw_toks) < 2:
        continue

    inf_form = kw_toks[1]
    inf_a_kwic, inf_b_kwic = kw_spans[1]

    left_plus_kwic = smart_concat(left_part, kwic_str)
    kwic_start = len(left_plus_kwic) - len(kwic_str)
    sentence = smart_concat(left_plus_kwic, right_part)

    inf_a_sent = kwic_start + inf_a_kwic
    inf_b_sent = kwic_start + inf_b_kwic

    masked = sentence[:inf_a_sent] + MASK_TOKEN + sentence[inf_b_sent:]

    rows.append({
        "sent_id": int(ridx),
        "sentence": sentence,
        "masked": masked,
        "inf_form": inf_form,
        "kwic": kwic_str
    })

clean = pd.DataFrame(rows).reset_index(drop=True)

print("Prepared rows:", len(clean))

clean = clean.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

if len(clean) > TARGET_N_SENT:
    clean = clean.iloc[:TARGET_N_SENT].copy()

print("Prepared rows after subsampling:", len(clean))

clean.head(15)

## Loading the collostructional gold ranking

In [ ]:
gold_raw = pd.read_csv(GOLD_CSV)

gold = gold_raw[["COLLEX", "COLL.STR.LOGL"]].copy()
gold.columns = ["lemma", "g2_score"]

gold["lemma"] = (
    gold["lemma"]
    .astype(str)
    .str.lower()
    .str.strip()
)

gold_lemmas = list(gold["lemma"])

lemma2id = {}

for lem in gold_lemmas:
    pieces = tok.tokenize(lem)

    if len(pieces) == 1:
        lemma2id[lem] = tok.convert_tokens_to_ids(pieces[0])

gold_eval = gold[gold["lemma"].isin(lemma2id)].copy()
gold_eval = gold_eval.reset_index(drop=True)

gold_list = list(gold_eval["lemma"])

print(
    "Gold total:", len(gold),
    "usable single-token lemmas:", len(gold_eval),
    "example:", gold_eval.head(10).to_dict(orient="records")
)

## Tokenization coverage across models

This diagnostic step compares candidate masked language models in terms of how many gold infinitive lemmas are represented as single tokens by each tokenizer.

This is relevant because the masked prediction task evaluates one masked position: multi-token infinitives cannot be directly compared to the model probability assigned to a single `[MASK]` token.

In [ ]:
model_names = {
    "BERT-base-italian-cased": "dbmdz/bert-base-italian-cased",
    "BERT-xxl-italian-cased": "dbmdz/bert-base-italian-xxl-cased",
    "BERT-base-italian-uncased": "dbmdz/bert-base-italian-uncased",
    "BERT-xxl-italian-uncased": "dbmdz/bert-base-italian-xxl-uncased",
    "UmBERTo": "Musixmatch/umberto-commoncrawl-cased-v1",
    "mBERT": "google-bert/bert-base-multilingual-uncased",
    "BERTino": "indigo-ai/BERTino",
    "ELECTRA-xxl-italian": "dbmdz/electra-base-italian-xxl-cased-generator",
}

results = []

for label, model_id in model_names.items():
    print(f"Loading tokenizer for {label}...")

    tok_tmp = AutoTokenizer.from_pretrained(model_id)
    single_token_lemmas = []
    multi_token_lemmas = []

    for lem in gold_lemmas:
        tokens = tok_tmp.tokenize(lem)

        if len(tokens) == 1:
            single_token_lemmas.append(lem)
        else:
            multi_token_lemmas.append((lem, tokens))

    total = len(gold_lemmas)
    usable = len(single_token_lemmas)
    ratio = usable / total if total else 0.0

    results.append({
        "model": label,
        "huggingface_id": model_id,
        "usable_single_tokens": usable,
        "total_lemmas": total,
        "coverage_ratio": ratio
    })

    print(f"{label}: {usable}/{total} ({ratio * 100:.1f}%) single-token lemmas")
    print("Example splits:", multi_token_lemmas[:5])
    print("-" * 80)

df_results = (
    pd.DataFrame(results)
    .sort_values("coverage_ratio", ascending=False)
    .reset_index(drop=True)
)

display(df_results)

best_model = df_results.iloc[0]["huggingface_id"]

print(f"Best model by single-token coverage: {best_model}")

## Open-vocabulary prediction

In [ ]:
vocab_size = mlm.get_input_embeddings().weight.shape[0]
sum_probs_full = np.zeros(vocab_size, dtype=np.float64)
N_used = 0

def ids_to_token_str(tid: int) -> str:
    """Convert token id to readable token string."""
    try:
        tok_str = tok.convert_ids_to_tokens([tid])[0]
        return tok.convert_tokens_to_string([tok_str])
    except Exception:
        return f"<UNK_{tid}>"


filled_rows = []

for i in tqdm(range(0, len(clean), BATCH_SIZE)):
    batch = clean.iloc[i:i + BATCH_SIZE]

    enc = tok(
        batch["masked"].tolist(),
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        logits = mlm(**enc).logits

    mask_pos = (enc["input_ids"] == tok.mask_token_id).nonzero(as_tuple=False)

    by_b = {}

    for b, pos in mask_pos:
        by_b.setdefault(int(b), []).append(int(pos))

    for b in range(logits.size(0)):
        poss = by_b.get(b, [])

        if len(poss) != 1:
            continue

        p = torch.softmax(logits[b, poss[0], :], dim=-1).detach().cpu().numpy()

        sum_probs_full += p
        N_used += 1

        tid = int(np.argmax(p))
        p1 = float(p[tid])
        token = ids_to_token_str(tid)

        masked = batch["masked"].iloc[b]
        filled = masked.replace(MASK_TOKEN, token, 1)

        filled_rows.append({
            "sent_id": int(batch["sent_id"].iloc[b]),
            "sentence": batch["sentence"].iloc[b],
            "masked": masked,
            "pred_token": token,
            "pred_prob": p1,
            "filled_sentence": filled
        })

avg_probs_full = sum_probs_full / max(N_used, 1)

eval_probs = {
    lem: float(avg_probs_full[tid])
    for lem, tid in lemma2id.items()
}

print("Averaged over sentences:", N_used)

filled_df = pd.DataFrame(filled_rows)
filled_df.to_csv(OUT_FILLED, index=False)

print("Saved filled sentences to", OUT_FILLED)

In [ ]:
vocab_tokens = []
vocab_strings = []

for i in range(vocab_size):
    try:
        t = tok.convert_ids_to_tokens(i)
    except Exception:
        t = None

    if t is None:
        t = f"<UNK_{i}>"

    try:
        s = tok.convert_tokens_to_string([t])
    except Exception:
        s = str(t)

    vocab_tokens.append(t)
    vocab_strings.append(s)

full_rank_df = pd.DataFrame({
    "token": vocab_strings,
    "token_id": range(vocab_size),
    "avg_prob": avg_probs_full
}).sort_values("avg_prob", ascending=False).reset_index(drop=True)

full_rank_df.to_csv(OUT_RANK_FULL, index=False)

print(f"Saved full-vocabulary ranking to {OUT_RANK_FULL} ({vocab_size} entries)")

## Open-vocabulary evaluation

In the open-vocabulary setting, the model predictions are averaged over the full vocabulary. This reflects the model's unconstrained preferences at the masked infinitive position.

The resulting model ranking is compared with the full collostructional gold ranking using Rank-Biased Overlap (RBO), after minimal normalization of common truncated infinitive forms.

In [ ]:
def load_full_vocab_df():
    """Load the full-vocabulary ranking from memory or file."""
    try:
        return full_rank_df.copy()
    except NameError:
        return pd.read_csv(OUT_RANK_FULL)


df_full = load_full_vocab_df()

model_full_list = (
    df_full
    .sort_values("avg_prob", ascending=False)["token"]
    .astype(str)
    .str.strip()
    .tolist()
)

gold_full_list = (
    gold["lemma"]
    .astype(str)
    .str.strip()
    .str.lower()
    .tolist()
)


def dedup_keep_first(seq):
    """Deduplicate ranking while preserving order."""
    out = []
    seen = set()

    for x in seq:
        x = "" if x is None else str(x).strip()

        if not x or x in seen:
            continue

        seen.add(x)
        out.append(x)

    return out

In [ ]:
CLITIC_SUFFIXES = (
    "melo", "mela", "meli", "mele", "telo", "tela", "teli", "tele",
    "celo", "cela", "celi", "cele", "velo", "vela", "veli", "vele",
    "glielo", "gliela", "glieli", "gliele",
    "mi", "ti", "si", "ci", "vi", "lo", "la", "li", "le", "ne", "gli"
)

BASE_TRUNC_MAP = {
    "far": "fare",
    "andar": "andare",
    "tener": "tenere",
    "esser": "essere",
    "aver": "avere",
    "dir": "dire",
    "dar": "dare",
    "star": "stare",
}

def normalize_token_for_rbo(t: str) -> str:
    """Normalize common truncated infinitive forms before RBO evaluation."""
    t = "" if t is None else str(t).strip()

    if not t:
        return t

    if t in BASE_TRUNC_MAP:
        return BASE_TRUNC_MAP[t]

    for base, lemma in BASE_TRUNC_MAP.items():
        if t.startswith(base):
            rest = t[len(base):]

            if rest in CLITIC_SUFFIXES:
                return lemma

    return t

In [ ]:
model_full_list_norm = [
    normalize_token_for_rbo(x)
    for x in model_full_list
]

model_full_rank_norm = dedup_keep_first(model_full_list_norm)
gold_full_rank = dedup_keep_first(gold_full_list)

rbo_open = rbo_ext_score(model_full_rank_norm, gold_full_rank, p=RBO_P)

print(f"[OPEN-VOCAB] RBO(model_full_vocab vs gold_full) with p={RBO_P}: {rbo_open:.4f}")


def mass_at_m_open(m):
    """Sum average probabilities assigned to the top-m gold lemmas."""
    topm = set(gold_list[:m])

    return float(
        sum(
            float(avg_probs_full[lemma2id[lem]])
            for lem in topm
            if lem in lemma2id
        )
    )


for m in (5, 10, 20, 50, 100):
    print(f"[OPEN-VOCAB] Mass@{m}: {mass_at_m_open(m):.4f}")

## Inspecting rankings and example predictions

In [ ]:
print(model_full_rank_norm[:20])
print(gold_full_rank[:20])

In [ ]:
try:
    examples_df = pd.read_csv(OUT_FILLED)
    print("Examples preview from", OUT_FILLED)
    display(examples_df.head(5))
except Exception as e:
    print("Could not load examples file:", e)

Examples preview from forward_volere_filled.csv


,sent_id,sentence,masked,pred_token,pred_prob,filled_sentence
0,2570,Perdonami non volevo essere autoritaria ma sa...,Perdonami non volevo [MASK] autoritaria ma sa...,sembrare,0.726346,Perdonami non volevo sembrare autoritaria ma ...
1,6009,io facesse una pause grande grande ai ai ragaz...,io facesse una pause grande grande ai ai ragaz...,dormire,0.068853,io facesse una pause grande grande ai ai ragaz...
2,256,La verdura colta direttamente nell''orto e po...,La verdura colta direttamente nell''orto e po...,parlare,0.405957,La verdura colta direttamente nell''orto e po...
3,3827,Se vuoi comprare Cinque Campi Lambrusco Dell ...,Se vuoi [MASK] Cinque Campi Lambrusco Dell Em...,assaggiare,0.276911,Se vuoi assaggiare Cinque Campi Lambrusco Del...
4,3462,"La scelta ""pro-Stato"" deve si essere assoluta...","La scelta ""pro-Stato"" deve si essere assoluta...",aderire,0.651737,"La scelta ""pro-Stato"" deve si essere assoluta..."


In [ ]:
TOP_N = 50

df_full = load_full_vocab_df()

total_mass = df_full["avg_prob"].sum()

if total_mass and total_mass > 0:
    df_full["avg_prob_norm"] = df_full["avg_prob"] / total_mass
else:
    df_full["avg_prob_norm"] = df_full["avg_prob"]

topN = df_full.head(TOP_N).loc[:, ["token", "avg_prob_norm", "avg_prob", "token_id"]]

TOPN_PATH = "topN_open_vocab.csv"
topN.to_csv(TOPN_PATH, index=False)

print(f"Saved Top-{TOP_N} open-vocab tokens to {TOPN_PATH}")
display(topN)

Saved Top-50 open-vocab tokens to topN_open_vocab.csv


,token,avg_prob_norm,avg_prob,token_id
0,sapere,0.026674,0.026675,1859
1,dire,0.018408,0.018408,442
2,fare,0.018251,0.018252,553
3,essere,0.009903,0.009903,387
4,andare,0.009870,0.009870,1184
5,dare,0.009432,0.009432,2209
6,vedere,0.009111,0.009111,1425
7,parlare,0.008479,0.008479,1691
8,provare,0.007732,0.007732,4865
9,ringraziare,0.007634,0.007634,9881


In [ ]:
print("MODEL TOP30:", model_full_rank_norm[:30])
print("GOLD TOP30:", gold_full_rank[:30])
print("Overlap TOP30:", set(model_full_rank_norm[:30]) & set(gold_full_rank[:30]))

MODEL TOP30: ['sapere', 'dire', 'fare', 'essere', 'andare', 'dare', 'vedere', 'parlare', 'provare', 'ringraziare', 'ricordare', 'avere', 'mettere', 'sottolineare', 'diventare', 'capire', 'chiedere', 'saperne', 'cambiare', 'conoscere', 'tornare', 'continuare', 'raccontare', 'entrare', 'chiederti', 'acquistare', 'spendere', 'aggiungere', 'usare', 'portare']
GOLD TOP30: ['sapere', 'dire', 'fare', 'ringraziare', 'vedere', 'dare', 'provare', 'chiedere', 'ricordare', 'approfondire', 'sottolineare', 'parlare', 'capire', 'mettere', 'rinunciare', 'condividere', 'segnalare', 'cambiare', 'partecipare', 'evitare', 'spendere', 'raccontare', 'cimentare', 'conoscere', 'comprare', 'acquistare', 'tornare', 'intraprendere', 'imparare', 'scopare']
Overlap TOP30: {'ricordare', 'vedere', 'raccontare', 'chiedere', 'conoscere', 'ringraziare', 'sottolineare', 'sapere', 'tornare', 'parlare', 'acquistare', 'dire', 'cambiare', 'spendere', 'mettere', 'dare', 'fare', 'capire', 'provare'}


## Restricted evaluation on single-token gold lemmas

This evaluation restricts the comparison to infinitive lemmas that are both present in the gold ranking and represented as a single token by the tokenizer. This is necessary because the masked language model predicts one token at the masked position: multi-token lemmas cannot be directly aligned with a single-token prediction.

Within this restricted set, model predictions are compared to the gold ranking using Spearman rank correlation.

In [ ]:
sub = gold_eval[["lemma"]].copy()
sub["lemma"] = sub["lemma"].astype(str).str.strip().str.lower()

mp = pd.DataFrame([
    {"lemma": lem, "model_avg_prob": eval_probs[lem]}
    for lem in eval_probs
])

mp["lemma"] = mp["lemma"].astype(str).str.strip().str.lower()

both = sub.merge(mp, on="lemma", how="inner").reset_index(drop=True)

print("Restricted subset size:", len(both))

both["gold_rank"] = np.arange(1, len(both) + 1)
both["model_rank"] = both["model_avg_prob"].rank(ascending=False, method="average")

rho = float(
    np.corrcoef(
        both["gold_rank"].values,
        both["model_rank"].values
    )[0, 1]
)

print(f"[RESTRICTED] Spearman rho (gold_rank vs model_rank): {rho:.4f}")

Restricted subset size (common single-token lemmas): 630
[RESTRICTED] Spearman rho (gold_rank vs model_rank): 0.5775


## Saving rankings and metrics

In [ ]:
rank_model = sorted(eval_probs.items(), key=lambda x: x[1], reverse=True)

rank_df = pd.DataFrame(rank_model, columns=["lemma", "avg_prob"])
rank_df["rank"] = np.arange(1, len(rank_df) + 1)

rank_df.to_csv(OUT_RANK, index=False)

gold_sub = gold_eval[["lemma"]].copy()
gold_sub["lemma"] = gold_sub["lemma"].astype(str).str.strip().str.lower()

model_sub = rank_df.copy()
model_sub["lemma"] = model_sub["lemma"].astype(str).str.strip().str.lower()
model_sub = model_sub.rename(columns={"avg_prob": "model_avg_prob"})

both = (
    gold_sub
    .merge(model_sub[["lemma", "model_avg_prob"]], on="lemma", how="inner")
    .reset_index(drop=True)
)

if len(both) > 1:
    both["gold_rank"] = np.arange(1, len(both) + 1)
    both["model_rank"] = both["model_avg_prob"].rank(ascending=False, method="average")

    spearman_restricted = float(
        np.corrcoef(
            both["gold_rank"].values,
            both["model_rank"].values
        )[0, 1]
    )
else:
    spearman_restricted = None

scores = {
    "RBO_open_fullvocab": float(rbo_open) if "rbo_open" in globals() else None,
    "RBO_p": RBO_P,
    "model": MLM_NAME,
    "modal": "volere",
    "N_sentences_used": int(N_used),
    "Spearman_restricted_rank": spearman_restricted,
}

for m in (5, 10, 20, 50, 100):
    scores[f"Mass_open@{m}"] = (
        float(mass_at_m_open(m))
        if "mass_at_m_open" in globals()
        else None
    )

with open(OUT_SCORES, "w", encoding="utf-8") as f:
    json.dump(scores, f, indent=2, ensure_ascii=False)

print("Saved restricted-subset ranking to", OUT_RANK)
print("Saved metrics to", OUT_SCORES)

rank_df.head(10)

Saved restricted-subset rank to forward_volere_ranking.csv and metrics to forward_volere_scores.json


,lemma,avg_prob,rank
0,sapere,0.026675,1
1,dire,0.018408,2
2,fare,0.018252,3
3,essere,0.009903,4
4,andare,0.009870,5
5,dare,0.009432,6
6,vedere,0.009111,7
7,parlare,0.008479,8
8,provare,0.007732,9
9,ringraziare,0.007634,10


## Logging model comparison

In [ ]:
model_log = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "model_name": MLM_NAME,
    "usable_lemmas": len(gold_eval),
    "gold_total": len(gold),
    "coverage_ratio": len(gold_eval) / len(gold) if len(gold) else 0.0,
    "rbo_score": rbo_open
}

log_path = "model_comparison_log.csv"
file_exists = os.path.exists(log_path)

with open(log_path, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(model_log.keys()))

    if not file_exists:
        writer.writeheader()

    writer.writerow(model_log)

print(f"Logged results for {MLM_NAME} to {log_path}")

summary_df = pd.read_csv(log_path).sort_values("rbo_score", ascending=False)

display(summary_df)

best = summary_df.iloc[0]

print(
    f"Current best model: {best['model_name']} "
    f"with RBO={best['rbo_score']:.4f}, "
    f"coverage={best['coverage_ratio'] * 100:.1f}% "
    f"({int(best['usable_lemmas'])}/{int(best['gold_total'])})"
)

## RBO sensitivity analysis

In [ ]:
p_grid = np.round(np.arange(0.01, 1.00, 0.01), 2)

rbo_vals = []

for p in p_grid:
    r = rbo_ext_score(model_full_rank_norm, gold_full_rank, p=float(p))
    rbo_vals.append(r)

p_sweep_df = pd.DataFrame({
    "p": p_grid,
    "rbo": rbo_vals
})

display(p_sweep_df)

best_idx = int(p_sweep_df["rbo"].values.argmax())
p_max = float(p_sweep_df.loc[best_idx, "p"])
rbo_max = float(p_sweep_df.loc[best_idx, "rbo"])

print(f"Best in grid: p_max={p_max:.2f} with RBO={rbo_max:.4f}")

plt.figure()
plt.plot(p_sweep_df["p"], p_sweep_df["rbo"])
plt.xlabel("p (RBO weighting parameter)")
plt.ylabel("RBO")

plt.scatter([p_max], [rbo_max])

plt.savefig("rbo_p_range_analysis.png", dpi=300, bbox_inches="tight")
plt.savefig("rbo_p_range_analysis.pdf", bbox_inches="tight")

plt.show()

In [ ]:
for k in [10, 20, 50, 100, 200, 500, 1000]:
    a = set(model_full_rank_norm[:k])
    b = set(gold_full_rank[:k])
    print(k, len(a & b))

10 7
20 14
50 33
100 60
200 112
500 256
1000 629
